# Multi Vector Retriever — 요약으로 검색, 원문으로 반환

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **MultiVectorRetriever**의 "검색용 표현"과 "반환용 문서"를 분리하는 전략 이해하기
- LLM으로 문서 요약을 생성하고 벡터스토어에 저장하는 방법 익히기
- `InMemoryStore`에 원문을 저장하고 연결하는 구조 구현하기

## 사전 지식

- `uuid`로 고유 ID를 생성하는 방법
- ParentDocumentRetriever의 부모-자식 문서 개념

---

## 전략 비교

### 전략 1: 요약 기반 검색
원문이 복잡하고 길 때 명확한 요약으로 검색 정확도를 높여요:

```
요약(검색용): "Word2Vec은 단어를 벡터로 변환하는 기법"
원문(반환용): "Word2Vec은... (전체 2000자 상세 설명)"
```

### 전략 2: 가상 질문 기반 검색
문서에서 예상되는 질문을 미리 생성해 저장하면 실제 사용자 질문과 더 잘 매칭돼요:

```
가상 질문: "Transformer란 무엇인가요?"
원문(반환용): "Transformer는 Attention 메커니즘을 사용..."
```

> **실무 팁**: 문서 수가 적을 때(5~10개)는 LLM 요약 비용이 크지 않지만, 수천 개 문서라면 비용을 사전에 계산해보세요.

In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
# ---------------------------------------------------
# 문서 요약을 생성하고 MultiVectorRetriever 구조 설정
# ---------------------------------------------------

# ============================================================
# TODO: ChatOpenAI로 각 문서의 요약을 생성하세요 (처음 5개 청크만)
# 힌트: chain.invoke({"doc": d.page_content}) — 한 문장 요약
# 예상 결과: 5개 요약 생성 완료 메시지 출력
# ============================================================

from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.stores import InMemoryStore
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

# 문서 로드 및 분할 (처음 5개만 사용 — LLM 요약 비용 절감)
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)
docs = text_splitter.split_documents(documents)[:5]

# LLM 및 요약 체인
# temperature=0: 요약이 일관되게 생성되도록 설정
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")
chain = (
    ChatPromptTemplate.from_template("다음 문서를 한 문장으로 요약하세요:\n\n{doc}")
    | llm
    | StrOutputParser()
)

# 요약 생성 (검색용)


print(f"✅ {len(docs)}개 문서의 요약 생성 완료")

In [ ]:
# ---------------------------------------------------
# MultiVectorRetriever 생성 — 요약을 검색용, 원문을 반환용으로 분리 저장
# ---------------------------------------------------

# ============================================================
# TODO: MultiVectorRetriever를 생성하고 요약(vectorstore)과 원문(docstore)을 연결하세요
# 힌트: uuid로 문서 ID를 생성하고 metadata에 id_key로 연결
# 예상 결과: vectorstore에 요약 저장, docstore에 원문 저장 완료 메시지 출력
# ============================================================

# MultiVectorRetriever 생성
# vectorstore: 요약 임베딩 저장 (검색용)
vectorstore = FAISS.from_texts(["init"], OpenAIEmbeddings(model="text-embedding-3-small"))
# InMemoryStore: 원문 저장 (반환용)
store = InMemoryStore()
# id_key: 요약과 원문을 연결하는 메타데이터 키
id_key = "doc_id"



# 문서 ID 생성 — uuid로 각 문서에 고유 ID 부여
doc_ids = [str(uuid.uuid4()) for _ in docs]

# 요약을 vectorstore에 저장 (검색용)
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

multi_vector_retriever.vectorstore.add_documents(summary_docs)
# 원문을 docstore에 저장 (반환용) — id를 키로 연결
multi_vector_retriever.docstore.mset(list(zip(doc_ids, docs)))

print("✅ MultiVectorRetriever 생성 완료")
print("  - 요약: vectorstore (검색용)")
print("  - 원문: docstore (반환용)")

In [ ]:
# 검색
query = "기계 학습 라이브러리에 대해 알려주세요"
retrieved_docs = multi_vector_retriever.invoke(query)

print(f"📝 쿼리: {query}\n")
print("="*80)
print("[MultiVectorRetriever 결과]")
print("="*80)
for i, doc in enumerate(retrieved_docs, 1):
    print(f"[문서 {i}]")
    print(f"길이: {len(doc.page_content)}자 ← 원문 (상세)")
    print(f"내용: {doc.page_content[:150]}...")
    print("-"*80)

print("\n💡 장점:")
print("  - 요약으로 검색 → 정확하고 빠름")
print("  - 원문 반환 → 상세한 정보")
print("  - 가상 질문으로도 활용 가능")


---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | 임베딩 대상(요약/가상질문)과 반환 대상(원본)을 분리 |
| 두 가지 전략 | 요약 기반 — 긴 문서에 적합 / 가상 질문 기반 — QA 데이터셋에 적합 |
| 저장 구조 | VectorStore에 요약/질문, InMemoryStore에 원본 문서 |
| 연결 키 | `uuid` — VectorStore의 메타데이터와 InMemoryStore의 키를 연결 |
| 전처리 비용 | 인덱싱 시 LLM 호출 필요 (요약 생성 단계) |

### 전략 선택 가이드

| 전략 | 적합한 상황 | 인덱싱 비용 |
|------|-------------|-------------|
| 요약 기반 (Summarization) | 문서가 길고 핵심만 검색할 때 | 높음 (LLM 요약 필요) |
| 가상 질문 (Hypothetical Questions) | 예상 질문이 명확한 FAQ형 데이터 | 높음 (LLM 생성 필요) |
| 소형 청크 (Small Chunks) | 빠른 검색이 우선일 때 | 낮음 |

### 다음 노트북 예고

**08-SelfQueryRetriever** — 자연어 질문에서 메타데이터 필터(연도, 카테고리, 평점 등)를 LLM이 자동으로 추출해 구조화된 검색을 수행하는 방법을 배워요.